# Assignment 1, FIT5201, S1 2026
- Name: Yin Zhang
- ID: [Student ID removed]

## Ridge Regression

### Question 4 Ridge Regression

[Original assignment diagram omitted from this portfolio copy.]

[Original assignment diagram omitted from this portfolio copy.]

### Implementation Discussion

The RidgeRegression class follows the standard structure with `__init__`, `fit`, and `predict`.

The model is trained using gradient descent based on the derived gradients:
- grad_w = (1/n) X^T (Xw + b − y) + lambda * w  
- grad_b = (1/n) sum(Xw + b − y)

The intercept term is not regularised, as it is updated separately without the penalty term.

Numerical stability is ensured by using gradient descent instead of matrix inversion and by applying a small learning rate, which allows stable optimisation even for small values of lambda.

In [ ]:
import numpy as np

class RidgeRegression:
    """
    Ridge Regression via (Stochastic) Gradient Descent.

    Parameters
    ----------
    lambda_reg : float
        Regularisation strength (lambda >= 0).
    lr : float
        Learning rate.
    n_iters : int
        Number of gradient descent iterations.
    batch_size : int or None
        If None, use full-batch gradient descent.
        If int, use stochastic (mini-batch) gradient descent.
    fit_intercept : bool
        Whether to fit a separate intercept term (NOT regularised).
    """

    def __init__(self, lambda_reg=1.0, lr=0.01, n_iters=1000,
                 batch_size=None, fit_intercept=True):
        self.lambda_reg = lambda_reg
        self.lr = lr
        self.n_iters = n_iters
        self.batch_size = batch_size
        self.fit_intercept = fit_intercept
        self.w_ = None   # weights (excludes intercept)
        self.b_ = None   # intercept

    def fit(self, X, y):
        """
        Fit the model.

        The intercept is treated separately and is NOT penalised,
        following standard Ridge Regression convention.
        """
        n, d = X.shape

        # Initialise parameters
        self.w_ = np.zeros(d)
        self.b_ = 0.0

        for iteration in range(self.n_iters):
            # ---- choose mini-batch or full batch ----
            if self.batch_size is None:
                X_batch, y_batch = X, y
                n_batch = n
            else:
                idx = np.random.choice(n, size=self.batch_size, replace=False)
                X_batch = X[idx]
                y_batch = y[idx]
                n_batch = self.batch_size

            # ---- compute residuals ----
            residuals = X_batch @ self.w_ + self.b_ - y_batch  # (n_batch,)

            # ---- gradients ----
            # grad w.r.t. w: (1/n_batch) * X^T r + lambda * w
            grad_w = (X_batch.T @ residuals) / n_batch + self.lambda_reg * self.w_

            # grad w.r.t. b: (1/n_batch) * sum(r) [NOT regularised]
            grad_b = residuals.mean() if self.fit_intercept else 0.0

            # ---- parameter update ----
            self.w_ -= self.lr * grad_w
            if self.fit_intercept:
                self.b_ -= self.lr * grad_b

        return self

    def predict(self, X):
        """Return predicted values for X."""
        return X @ self.w_ + self.b_

### Experimental Setup Discussion

The experiment evaluates Ridge Regression under different values of the regularisation parameter lambda.

A fixed large test set is used to obtain a reliable estimate of generalisation performance, while multiple independent training sets (N=20) are sampled to reduce randomness and estimate average behaviour.

Polynomial features of degree 4 are used to introduce model flexibility, allowing the effect of regularisation to be clearly observed.

The target variable is centred before training, which effectively acts as an unregularised intercept term. This ensures consistency with standard Ridge Regression, where the intercept is not penalised.

The use of a logarithmic grid for lambda enables exploration across several orders of magnitude, capturing both underfitting large lambda and overfitting small lambda regimes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# hyper-parameters
N_TRAIN = 20          # training set size (per repeat)
N_TEST = 5000         # large independent test set
N_REPEATS = 30        # number of independent experiments
DEGREE = 4            # polynomial degree
LAMBDAS = np.geomspace(1e-10, 1e-1, 60)  # geometric grid


# helper: polynomial feature matrix (no intercept column)
def poly_features(x, degree):
    return np.column_stack([x ** k for k in range(1, degree + 1)])


# data generator
def generate_data(n, rng):
    x = rng.uniform(-0.6, 0.6, n)
    y = np.sin(2 * np.pi * x) ** 2 + 3 * x + rng.normal(0, 0.1, n)
    return x, y


def ridge_closed_form(X_tr, y_tr, lam):
    n, d = X_tr.shape
    A = X_tr.T @ X_tr + n * lam * np.eye(d)
    w = np.linalg.solve(A, X_tr.T @ y_tr)
    return w


def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# fixed large test set
x_test, y_test = generate_data(N_TEST, rng)
X_test = poly_features(x_test, DEGREE)

# sweep over lambda
train_errors = np.zeros((N_REPEATS, len(LAMBDAS)))
test_errors = np.zeros((N_REPEATS, len(LAMBDAS)))

for r in range(N_REPEATS):
    x_tr, y_tr = generate_data(N_TRAIN, rng)
    X_tr = poly_features(x_tr, DEGREE)

    # centre y by its mean (acts as unregularised intercept)
    y_mean = y_tr.mean()
    y_tr_c = y_tr - y_mean
    y_te_c = y_test - y_mean

    for j, lam in enumerate(LAMBDAS):
        w = ridge_closed_form(X_tr, y_tr_c, lam)
        train_errors[r, j] = mse(y_tr_c, X_tr @ w)
        test_errors[r, j] = mse(y_te_c, X_test @ w)

mean_train = train_errors.mean(axis=0)
mean_test = test_errors.mean(axis=0)

best_idx = np.argmin(mean_test)
best_lambda = LAMBDAS[best_idx]

print(f"Best λ: {best_lambda:.2e}")
print(f"→ Mean train MSE at best λ: {mean_train[best_idx]:.4f}")
print(f"→ Mean test  MSE at best λ: {mean_test[best_idx]:.4f}")
print(f"→ Mean train MSE at λ=1e-10 (min): {mean_train[0]:.4f}")
print(f"→ Mean test  MSE at λ=1e-10 (min): {mean_test[0]:.4f}")
print(f"→ Mean train MSE at λ=1e-1  (max): {mean_train[-1]:.4f}")
print(f"→ Mean test  MSE at λ=1e-1  (max): {mean_test[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(LAMBDAS, mean_train, label="Training error", lw=2)
ax.loglog(LAMBDAS, mean_test, label="Test error", linestyle="--", lw=2)
ax.axvline(best_lambda, linestyle=":", lw=2, label=f"Best λ = {best_lambda:.2e}")

ax.set_xlabel(r"$\lambda$ (log scale)", fontsize=13)
ax.set_ylabel("MSE (log scale)", fontsize=13)
ax.set_title(
    "Ridge Regression: Training vs Test Error\n"
    rf"(degree-4 polynomial, $n_{{\mathrm{{train}}}}={N_TRAIN}$, averaged over {N_REPEATS} repeats)",
    fontsize=12
)
ax.legend(fontsize=12)
ax.grid(True, which="both", ls=":")

plt.tight_layout()
plt.savefig("ridge_bias_variance.pdf")
plt.show()

### Analysis of the Plot

At very small values of lambda, the training error is low while the test error is noticeably higher, indicating overfitting. In this regime, the model behaves similarly to ordinary least squares and is sensitive to noise in the small training set.

As lambda increases, the regularisation term reduces model variance, leading to a decrease in test error. The best performance is achieved at an intermediate value of lambda (approximately 1.48 × 10^-3), which represents the optimal bias–variance trade-off.

For large lambda, both training and test errors increase significantly. This indicates underfitting, as the model becomes overly constrained and unable to capture the underlying structure of the data.

Overall, the results clearly demonstrate the bias–variance trade-off:
- small lambda → low bias, high variance (overfitting)
- large lambda → high bias, low variance (underfitting)
- intermediate lambda → best generalisation performance

Additionally, the training error increases monotonically with lambda, meaning that model selection cannot rely on training error alone and must consider test performance.